# Cuadernillo MIMIC-III: integracion, diccionario y preparacion para ML

Este cuadernillo esta disenado para:

1. Entender de que trata el dataset MIMIC-III y sus tablas principales.
2. Construir una tabla de columnas por dataset con significado practico.
3. Integrar tablas clinicas en un dataset analitico.
4. Transformar variables a formato numerico y documentar que se transforma y por que.
5. Dejar listo un pipeline para ejercicios de regresion, clasificacion y redes neuronales.

> Nota: algunas tablas (por ejemplo `CHARTEVENTS`) son muy grandes. Este cuadernillo usa una estrategia de agregacion y seleccion de variables para mantener el flujo usable en un equipo personal.

## 1) Contexto general del dataset

**MIMIC-III** (Medical Information Mart for Intensive Care) es una base de datos clinica de pacientes de UCI que incluye:

- Datos demograficos (`PATIENTS`).
- Episodios de hospitalizacion (`ADMISSIONS`).
- Estancias en UCI (`ICUSTAYS`).
- Eventos clinicos temporales (monitorizacion, laboratorio, medicamentos, microbiologia, notas, procedimientos, etc.).

### Claves relacionales principales

- `subject_id`: paciente.
- `hadm_id`: ingreso hospitalario.
- `icustay_id`: estancia en UCI.

### Tipos de problemas de ML que permite

- **Clasificacion**: mortalidad hospitalaria (`hospital_expire_flag`), readmision, sepsis, etc.
- **Regresion**: duracion de estancia (LOS), consumo de recursos, costos proxy.
- **Modelos secuenciales / redes neuronales**: series temporales por hora en UCI (chartevents/labevents).

### Consideraciones tecnicas

- Datos longitudinales con timestamps.
- Fuerte presencia de valores faltantes.
- Variables categoricas y textuales extensas.
- Tablas muy grandes requieren agregacion, muestreo o procesamiento incremental.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

BASE_DIR = Path('.')
csv_files = sorted(BASE_DIR.glob('*.csv'))
len(csv_files), [f.name for f in csv_files[:8]]

## 2) Tabla de columnas por dataset (diccionario practico)

En esta seccion, para **cada CSV** se genera una tabla con:

- nombre de columna,
- tipo de variable inferido,
- descripcion (heuristica basada en nomenclatura MIMIC).

Esto sirve como un diccionario util para el analisis. Para un significado clinico formal al 100%, se recomienda contrastar con la documentacion oficial de MIMIC-III.

In [ ]:
# Diccionario base de descripciones por nombres comunes
desc_map = {
    'row_id': 'Identificador interno de la fila en la tabla.',
    'subject_id': 'Identificador unico del paciente.',
    'hadm_id': 'Identificador del ingreso hospitalario.',
    'icustay_id': 'Identificador de la estancia en UCI.',
    'itemid': 'Codigo del item clinico (se mapea con tablas D_*).',
    'cgid': 'Identificador del cuidador/profesional que registro el dato.',
    'charttime': 'Timestamp clinico del evento/medicion.',
    'storetime': 'Timestamp de almacenamiento del registro.',
    'starttime': 'Tiempo de inicio de evento.',
    'endtime': 'Tiempo de fin de evento.',
    'admittime': 'Fecha/hora de admision hospitalaria.',
    'dischtime': 'Fecha/hora de alta hospitalaria.',
    'deathtime': 'Fecha/hora de fallecimiento (si aplica).',
    'intime': 'Inicio de estancia en UCI.',
    'outtime': 'Fin de estancia en UCI.',
    'dob': 'Fecha de nacimiento.',
    'dod': 'Fecha de defuncion.',
    'gender': 'Sexo del paciente.',
    'age': 'Edad del paciente (derivada normalmente, no siempre viene directa).',
    'los': 'Length of stay (duracion de estancia).',
    'value': 'Valor textual original de la observacion.',
    'valuenum': 'Valor numerico de la observacion.',
    'valueuom': 'Unidad del valor.',
    'flag': 'Bandera del laboratorio (normal/anormal, etc.).',
    'label': 'Nombre legible del codigo/variable.',
    'category': 'Categoria del evento/nota/item.',
    'dbsource': 'Sistema de origen (CareVue/MetaVision).',
    'diagnosis': 'Diagnostico principal de admision/episodio.',
    'hospital_expire_flag': 'Indicador de mortalidad hospitalaria (target de clasificacion).',
    'admission_type': 'Tipo de admision (electiva, emergencia, etc.).',
    'admission_location': 'Origen de admision.',
    'discharge_location': 'Destino al alta.',
    'insurance': 'Tipo de seguro.',
    'language': 'Idioma del paciente.',
    'religion': 'Religion registrada.',
    'marital_status': 'Estado civil.',
    'ethnicity': 'Grupo etnico registrado.',
    'drug': 'Nombre del medicamento.',
    'route': 'Via de administracion del medicamento.',
    'text': 'Texto libre (notas clinicas).',
    'icd9_code': 'Codigo ICD-9 diagnostico/procedimiento.'
}

def infer_kind(col):
    c = col.lower()
    if c.endswith('time') or c in {'dob','dod','chartdate','startdate','enddate'}:
        return 'fecha/hora'
    if c.endswith('_id') or c in {'itemid','cgid','row_id'}:
        return 'identificador'
    if c in {'valuenum','los','hospital_expire_flag','expire_flag'}:
        return 'numerica'
    if c in {'value','label','category','diagnosis','drug','text','ethnicity'}:
        return 'texto/categorica'
    return 'mixta o por revisar'

def describe_col(col):
    c = col.lower()
    if c in desc_map:
        return desc_map[c]
    if c.endswith('_id'):
        return 'Identificador relacional para unir tablas.'
    if 'time' in c or c in {'chartdate','startdate','enddate'}:
        return 'Variable temporal del proceso clinico.'
    if c.startswith('is_') or c.endswith('_flag'):
        return 'Variable binaria indicadora.'
    return 'Descripcion inferida: revisar documentacion oficial para definicion exacta.'

schema_tables = {}
for f in csv_files:
    cols = pd.read_csv(f, nrows=0).columns.tolist()
    schema_df = pd.DataFrame({
        'columna': cols,
        'tipo_inferido': [infer_kind(c) for c in cols],
        'descripcion': [describe_col(c) for c in cols]
    })
    schema_tables[f.stem] = schema_df

for name, sdf in schema_tables.items():
    display(Markdown(f'### {name}'))
    display(sdf)


## 3) Estrategia de integracion para dataset analitico

Para entrenamiento clasico de ML, conviene construir una tabla por estancia en UCI (una fila por `icustay_id`) o por admision (`hadm_id`).

En este cuadernillo construiremos una base por `icustay_id`, uniendo:

- `PATIENTS` + `ADMISSIONS` + `ICUSTAYS` (nucleo demografico/episodio).
- Agregados de `LABEVENTS` (estadisticos por ingreso).
- Agregados de `PRESCRIPTIONS` (conteos de farmacos y vias).
- Agregados de codigos (`DIAGNOSES_ICD`, `PROCEDURES_ICD`).

Esto mantiene trazabilidad clinica y reduce dimensionalidad.

In [ ]:
# Carga de tablas core
patients = pd.read_csv('PATIENTS.csv', parse_dates=['dob','dod','dod_hosp','dod_ssn'])
adm = pd.read_csv('ADMISSIONS.csv', parse_dates=['admittime','dischtime','deathtime','edregtime','edouttime'])
icu = pd.read_csv('ICUSTAYS.csv', parse_dates=['intime','outtime'])

core = (
    icu.merge(adm, on=['subject_id','hadm_id'], how='left', suffixes=('_icu','_adm'))
       .merge(patients[['subject_id','gender','dob','dod','expire_flag']], on='subject_id', how='left')
)

# Edad al ingreso (controlando edades negativas por problemas de fecha)
core['age_years'] = (core['admittime'] - core['dob']).dt.days / 365.25
core.loc[core['age_years'] < 0, 'age_years'] = np.nan

# Duraciones en horas
core['icu_los_hours'] = (core['outtime'] - core['intime']).dt.total_seconds() / 3600.0
core['hosp_los_hours'] = (core['dischtime'] - core['admittime']).dt.total_seconds() / 3600.0

core.shape, core[['subject_id','hadm_id','icustay_id']].drop_duplicates().shape

In [ ]:
# Agregados de laboratorio por hadm_id
labs = pd.read_csv('LABEVENTS.csv', usecols=['hadm_id','itemid','valuenum'])
labs = labs.dropna(subset=['hadm_id'])

lab_agg = labs.groupby('hadm_id').agg(
    lab_n=('valuenum','count'),
    lab_mean=('valuenum','mean'),
    lab_std=('valuenum','std'),
    lab_min=('valuenum','min'),
    lab_max=('valuenum','max'),
    lab_item_unique=('itemid','nunique')
).reset_index()

# Agregados de prescripciones por hadm_id
rx = pd.read_csv('PRESCRIPTIONS.csv', usecols=['hadm_id','drug','route'])
rx = rx.dropna(subset=['hadm_id'])
rx_agg = rx.groupby('hadm_id').agg(
    rx_n=('drug','count'),
    rx_drug_unique=('drug','nunique'),
    rx_route_unique=('route','nunique')
).reset_index()

# Agregados de codigos diagnosticos/procedimientos por hadm_id
diag = pd.read_csv('DIAGNOSES_ICD.csv', usecols=['hadm_id','icd9_code'])
proc = pd.read_csv('PROCEDURES_ICD.csv', usecols=['hadm_id','icd9_code'])

diag_agg = diag.groupby('hadm_id').agg(diag_code_n=('icd9_code','nunique')).reset_index()
proc_agg = proc.groupby('hadm_id').agg(proc_code_n=('icd9_code','nunique')).reset_index()

# Merge final de agregados
df = (core
      .merge(lab_agg, on='hadm_id', how='left')
      .merge(rx_agg, on='hadm_id', how='left')
      .merge(diag_agg, on='hadm_id', how='left')
      .merge(proc_agg, on='hadm_id', how='left'))

df.shape

## 4) Transformaciones a numerico: que cambios se hacen y por que

A continuacion aplicamos transformaciones estandar para ML:

1. **Fechas -> variables numericas derivadas**
   - Ya derivamos `age_years`, `icu_los_hours`, `hosp_los_hours`.
   - Motivo: los modelos tabulares trabajan mejor con magnitudes numericas directamente interpretables.

2. **Texto/categoricas -> codificacion numerica**
   - One-hot para columnas de baja/media cardinalidad (`gender`, `admission_type`, etc.).
   - Motivo: evita imponer orden artificial y permite que el modelo use categorias de forma no ordinal.

3. **Valores faltantes**
   - Mediana para numericas, categoria mas frecuente para categoricas.
   - Motivo: mantener filas y evitar sesgos por eliminacion agresiva.

4. **Escalado numerico**
   - `StandardScaler` para modelos sensibles a escala (regresion logistica, redes neuronales).

5. **Reduccion de dimensionalidad indirecta**
   - En lugar de usar millones de filas crudas de eventos, agregamos estadisticos por admision.
   - Motivo: convertir series/eventos en resumenes robustos y computacionalmente tratables.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report, mean_absolute_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression

# Targets de ejemplo
target_cls = 'hospital_expire_flag'   # clasificacion binaria
target_reg = 'icu_los_hours'          # regresion

# Seleccion de columnas candidatas
drop_cols = [
    'row_id_icu','row_id_adm','subject_id','hadm_id','icustay_id',
    'dob','dod','dod_hosp','dod_ssn','admittime','dischtime','deathtime','intime','outtime',
    'edregtime','edouttime',
]

feature_df = df.copy()
existing_drop = [c for c in drop_cols if c in feature_df.columns]
feature_df = feature_df.drop(columns=existing_drop)

# Quitamos columnas de texto libre no controlado para baseline
for c in ['diagnosis']:
    if c in feature_df.columns:
        feature_df = feature_df.drop(columns=[c])

# Filtrar filas con target disponible
cls_df = feature_df.dropna(subset=[target_cls]).copy()
reg_df = feature_df.dropna(subset=[target_reg]).copy()

num_cols = cls_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = cls_df.select_dtypes(exclude=[np.number]).columns.tolist()
for t in [target_cls, target_reg]:
    if t in num_cols:
        num_cols.remove(t)
    if t in cat_cols:
        cat_cols.remove(t)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

len(num_cols), len(cat_cols), cls_df.shape, reg_df.shape

In [ ]:
## Baseline de clasificacion: mortalidad hospitalaria
X_cls = cls_df.drop(columns=[target_cls])
y_cls = cls_df[target_cls].astype(int)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

clf = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=200))
])

clf.fit(Xc_train, yc_train)
yc_pred = clf.predict(Xc_test)
yc_prob = clf.predict_proba(Xc_test)[:, 1]

print('AUC:', roc_auc_score(yc_test, yc_prob))
print(classification_report(yc_test, yc_pred))

In [ ]:
## Baseline de regresion: duracion de estancia UCI (horas)
X_reg = reg_df.drop(columns=[target_reg])
y_reg = reg_df[target_reg]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LinearRegression())
])

reg.fit(Xr_train, yr_train)
yr_pred = reg.predict(Xr_test)

print('MAE:', mean_absolute_error(yr_test, yr_pred))
print('R2 :', r2_score(yr_test, yr_pred))

## 5) Preparacion para redes neuronales

Para redes neuronales tabulares (MLP), puedes reutilizar el mismo `preprocess` y luego:

- convertir la matriz transformada a `float32`,
- entrenar un modelo denso en TensorFlow/PyTorch,
- usar early stopping,
- balancear clases si el target esta desbalanceado.

Ejemplo orientativo (no ejecuta entrenamiento pesado por defecto):

In [ ]:
# Ejemplo de conversion para red neuronal (clasificacion)
# Requiere: pip install tensorflow

# X_train_nn = preprocess.fit_transform(Xc_train)
# X_test_nn  = preprocess.transform(Xc_test)
# y_train_nn = yc_train.values.astype('float32')
# y_test_nn  = yc_test.values.astype('float32')

# Si el transformador devuelve sparse matrix:
# X_train_nn = X_train_nn.toarray().astype('float32')
# X_test_nn = X_test_nn.toarray().astype('float32')

# import tensorflow as tf
# model = tf.keras.Sequential([
#     tf.keras.layers.Input(shape=(X_train_nn.shape[1],)),
#     tf.keras.layers.Dense(256, activation='relu'),
#     tf.keras.layers.Dropout(0.3),
#     tf.keras.layers.Dense(64, activation='relu'),
#     tf.keras.layers.Dense(1, activation='sigmoid')
# ])
# model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
# model.fit(X_train_nn, y_train_nn, validation_split=0.2, epochs=20, batch_size=256)
# model.evaluate(X_test_nn, y_test_nn)

## 6) Resumen de transformaciones aplicadas (trazabilidad)

Transformaciones realizadas en este cuadernillo:

- Join por claves clinicas: `subject_id`, `hadm_id`, `icustay_id`.
- Fechas a duraciones/edad numerica: `age_years`, `icu_los_hours`, `hosp_los_hours`.
- Agregacion de eventos de alta frecuencia (labs, prescripciones, ICD) a estadisticos por admision.
- Imputacion de faltantes: mediana (numericas), moda (categoricas).
- Codificacion de categoricas: one-hot.
- Escalado de numericas: estandarizacion.

Con esto, la base queda lista para experimentar con modelos de regresion, clasificacion y redes neuronales tabulares.

## 7) Bitacora explicita de cambios de transformacion

Esta seccion deja una tabla clara de **que se transforma, como y por que**, para documentacion y reporte metodologico.

In [ ]:
transform_log = []

for c in num_cols:
    transform_log.append({
        'columna': c,
        'tipo_original': 'numerica',
        'transformacion': 'imputacion_mediana + escalado_estandar',
        'motivo': 'manejar faltantes y homogeneizar escala para modelos lineales/NN'
    })

for c in cat_cols:
    transform_log.append({
        'columna': c,
        'tipo_original': 'categorica/texto corto',
        'transformacion': 'imputacion_moda + one_hot_encoding',
        'motivo': 'convertir categorias a formato numerico sin orden artificial'
    })

# Variables derivadas de fechas y su justificacion
for c, why in [
    ('age_years', 'representar edad en formato numerico interpretable'),
    ('icu_los_hours', 'objetivo/feature continua para tareas de regresion'),
    ('hosp_los_hours', 'capturar severidad/consumo de recursos en horas')
]:
    if c in df.columns:
        transform_log.append({
            'columna': c,
            'tipo_original': 'derivada temporal',
            'transformacion': 'ingenieria_de_variables_desde_fechas',
            'motivo': why
        })

transform_log_df = pd.DataFrame(transform_log).sort_values(['tipo_original','columna']).reset_index(drop=True)
transform_log_df.head(30)

In [ ]:
display(transform_log_df)
print('Total de columnas documentadas:', len(transform_log_df))